# Model Selection & Evaluation — Customer Churn

This notebook walks through the same pipeline implemented in `src/model_selection.py`, with extra visual checks at each stage. It builds directly on the processed dataset from the feature-engineering task (`data/customer_churn_processed.csv`).

**Steps covered:**
1. Load the processed dataset
2. Train/test split
3. Train three candidate models (Logistic Regression, Decision Tree, Random Forest)
4. Evaluate with accuracy, precision, recall, F1-score
5. Cross-validate to check for overfitting
6. Compare & justify the best model

> **Note on "linear regression":** the task brief lists linear regression as an example algorithm, but `churn` is a binary target and the required metrics (accuracy/precision/recall/F1) are classification metrics. **Logistic Regression** is used as the linear model here — it's the linear-model equivalent for classification problems.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
%matplotlib inline

## 1. Load processed data

Already cleaned, feature-engineered, encoded, scaled, and reduced to the top 8 features by the feature-engineering task.

In [ ]:
df = pd.read_csv('../data/customer_churn_processed.csv')
X = df.drop(columns=['churn'])
y = df['churn']
print(X.shape)
df.head()

## 2. Train/test split

80/20 split, stratified on the target to preserve the churn rate in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print('Train churn rate:', y_train.mean().round(3))
print('Test churn rate:', y_test.mean().round(3))

## 3. Define candidate models

- **Logistic Regression** — linear baseline, fast, interpretable
- **Decision Tree** — captures non-linear splits, prone to overfitting
- **Random Forest** — ensemble of trees, usually more robust/accurate

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE),
}

## 4. Train & evaluate on the held-out test set

In [ ]:
holdout_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    holdout_results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
    }

pd.DataFrame(holdout_results).T.round(4)

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 5. Cross-validation

5-fold stratified cross-validation checks whether the holdout performance generalizes, rather than being a lucky/unlucky split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1']

cv_results = {}
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
    cv_results[name] = {f'{m}_mean': scores[f'test_{m}'].mean() for m in scoring}

pd.DataFrame(cv_results).T.round(4)

## 6. Compare models

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
for i, (name, res) in enumerate(holdout_results.items()):
    ax.bar(x + i * width, [res[m] for m in metrics], width, label=name)
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.set_title('Model comparison on held-out test set')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Conclusion

**Random Forest** has the highest mean cross-validated F1-score and the lowest variance across folds, making it the most reliable choice despite Logistic Regression scoring similarly on accuracy. See `reports/report.md` for the full write-up and justification.